# Formative 2: Multimodal User Authentication and Product Recommendation\n
\n
This notebook follows the assignment flow and rubric end-to-end.

## 1) Install Dependencies

In [ ]:
!pip -q install numpy pandas scikit-learn joblib opencv-python scikit-image librosa soundfile matplotlib seaborn scipy

## 2) Clone Repository and Move Into Project Folder

In [ ]:
import subprocess\n
from pathlib import Path\n
\n
REPO_URL = "https://github.com/Miranics/ML_Multimodal-User-Auth-System.git"\n
REPO_DIR = Path("/content/ML_Multimodal-User-Auth-System")\n
\n
if not REPO_DIR.exists():\n
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)\n
\n
%cd /content/ML_Multimodal-User-Auth-System\n
print("Working directory:", Path.cwd())

## 3) Optional: Upload Missing Data Files to Colab

In [ ]:
# Use this cell only if your data is not already in the repo.\n
# from google.colab import files\n
# uploaded = files.upload()\n
# Then move uploaded files into data/raw, data/images/<member>, and data/audio/<member>.

## 4) Import Libraries and Define Paths

In [ ]:
import os\n
import sys\n
import json\n
import numpy as np\n
import pandas as pd\n
import cv2\n
import seaborn as sns\n
import matplotlib.pyplot as plt\n
import librosa\n
import librosa.display\n
from pathlib import Path\n
\n
ROOT = Path.cwd()\n
RAW = ROOT / "data" / "raw"\n
IMAGES = ROOT / "data" / "images"\n
AUDIO = ROOT / "data" / "audio"\n
PROCESSED = ROOT / "data" / "processed"\n
REPORTS = ROOT / "reports"\n
\n
for p in [PROCESSED, REPORTS]:\n
    p.mkdir(parents=True, exist_ok=True)\n
\n
print("Paths initialized.")

## 5) Verify Assignment Inputs

In [ ]:
required_tabular = [\n
    RAW / "customer_social_profiles.csv",\n
    RAW / "customer_transactions.csv"\n
]\n
\n
for f in required_tabular:\n
    print(f"{f}:", f.exists())\n
\n
print("Image member folders:", [p.name for p in IMAGES.iterdir() if p.is_dir()])\n
print("Audio member folders:", [p.name for p in AUDIO.iterdir() if p.is_dir()])

## 6) Load Raw Tabular Data

In [ ]:
social = pd.read_csv(RAW / "customer_social_profiles.csv")\n
transactions = pd.read_csv(RAW / "customer_transactions.csv")\n
\n
print("Social shape:", social.shape)\n
print("Transactions shape:", transactions.shape)\n
display(social.head())\n
display(transactions.head())

## 7) Data Cleaning and Merge Validation Checks

In [ ]:
def quality_report(df, name):\n
    print(f"\n{name}\
,
Duplicates:", int(df.duplicated().sum()))\n
    print("Nulls per column:")\n
    print(df.isnull().sum())\n
    print("Dtypes:")\n
    print(df.dtypes)\n
\n
quality_report(social, "Social Profiles")\n
quality_report(transactions, "Transactions")

## 8) EDA: Summary Statistics

In [ ]:
display(social.describe(include="all"))\n
display(transactions.describe(include="all"))

## 9) EDA: Distribution, Outliers, and Correlation Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))\n
\n
sns.histplot(social["engagement_score"], kde=True, ax=axes[0], color="steelblue")\n
axes[0].set_title("Distribution: engagement_score")\n
\n
sns.boxplot(y=transactions["purchase_amount"], ax=axes[1], color="salmon")\n
axes[1].set_title("Outliers: purchase_amount")\n
\n
num_tx = transactions.select_dtypes(include=[np.number])\n
sns.heatmap(num_tx.corr(numeric_only=True), annot=True, cmap="Blues", ax=axes[2])\n
axes[2].set_title("Correlation: transaction numeric features")\n
\n
plt.tight_layout()\n
plt.show()

## 10) Run Tabular Merge and Engineered Features

In [ ]:
subprocess.run([sys.executable, "-m", "src.data_merge"], check=True)\n
merged = pd.read_csv(PROCESSED / "merged_features.csv")\n
print("Merged shape:", merged.shape)\n
display(merged.head())

## 11) Display Sample Face Images

In [ ]:
image_paths = sorted([p for p in IMAGES.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])\n
print("Total images:", len(image_paths))\n
\n
show_n = min(12, len(image_paths))\n
if show_n > 0:\n
    cols = 4\n
    rows = int(np.ceil(show_n / cols))\n
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))\n
    axes = np.array(axes).reshape(rows, cols)\n
\n
    for i, ax in enumerate(axes.flatten()):\n
        if i < show_n:\n
            p = image_paths[i]\n
            img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)\n
            ax.imshow(img)\n
            ax.set_title(f"{p.parent.name} - {p.name}")\n
            ax.axis("off")\n
        else:\n
            ax.axis("off")\n
\n
    plt.tight_layout()\n
    plt.show()

## 12) Preview Image Augmentations

In [ ]:
if len(image_paths) > 0:\n
    p = image_paths[0]\n
    img_bgr = cv2.imread(str(p))\n
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)\n
    rot = cv2.warpAffine(\n
        img_bgr,\n
        cv2.getRotationMatrix2D((img_bgr.shape[1] // 2, img_bgr.shape[0] // 2), 15, 1.0),\n
        (img_bgr.shape[1], img_bgr.shape[0])\n
    )\n
    flip = cv2.flip(img_bgr, 1)\n
\n
    variants = [\n
        ("Original", cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), None),\n
        ("Rotate +15", cv2.cvtColor(rot, cv2.COLOR_BGR2RGB), None),\n
        ("Horizontal Flip", cv2.cvtColor(flip, cv2.COLOR_BGR2RGB), None),\n
        ("Grayscale", gray, "gray")\n
    ]\n
\n
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))\n
    for ax, (title, im, cmap) in zip(axes, variants):\n
        ax.imshow(im, cmap=cmap)\n
        ax.set_title(title)\n
        ax.axis("off")\n
    plt.tight_layout()\n
    plt.show()

## 13) Run Image Feature Extraction

In [ ]:
subprocess.run([sys.executable, "-m", "src.image_pipeline"], check=True)\n
image_features = pd.read_csv(PROCESSED / "image_features.csv")\n
print("image_features.csv shape:", image_features.shape)\n
display(image_features.head())

## 14) Visualize Audio Waveform and Spectrogram

In [ ]:
audio_paths = sorted([p for p in AUDIO.rglob("*") if p.suffix.lower() in {".wav", ".mp3", ".flac", ".ogg"}])\n
print("Total audio files:", len(audio_paths))\n
\n
if len(audio_paths) > 0:\n
    sample_audio = audio_paths[0]\n
    y, sr = librosa.load(sample_audio, sr=None)\n
\n
    fig, axes = plt.subplots(2, 1, figsize=(12, 7))\n
    librosa.display.waveshow(y, sr=sr, ax=axes[0])\n
    axes[0].set_title(f"Waveform: {sample_audio.name}")\n
\n
    S = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)\n
    img = librosa.display.specshow(S, sr=sr, x_axis="time", y_axis="log", ax=axes[1])\n
    axes[1].set_title("Spectrogram")\n
    fig.colorbar(img, ax=axes[1], format="%+2.0f dB")\n
\n
    plt.tight_layout()\n
    plt.show()

## 15) Preview Audio Augmentations

In [ ]:
if len(audio_paths) > 0:\n
    y_base, sr_base = librosa.load(audio_paths[0], sr=None)\n
    y_base = y_base.astype(np.float32)\n
    y_noise = y_base + 0.005 * np.random.randn(len(y_base)).astype(np.float32)\n
    y_stretch = librosa.effects.time_stretch(y_base, rate=0.9)\n
    y_pitch = librosa.effects.pitch_shift(y_base, sr=sr_base, n_steps=2)\n
\n
    fig, axes = plt.subplots(4, 1, figsize=(12, 10))\n
    librosa.display.waveshow(y_base, sr=sr_base, ax=axes[0]); axes[0].set_title("Original")\n
    librosa.display.waveshow(y_noise, sr=sr_base, ax=axes[1]); axes[1].set_title("Noise")\n
    librosa.display.waveshow(y_stretch, sr=sr_base, ax=axes[2]); axes[2].set_title("Time Stretch")\n
    librosa.display.waveshow(y_pitch, sr=sr_base, ax=axes[3]); axes[3].set_title("Pitch Shift +2")\n
    plt.tight_layout()\n
    plt.show()

## 16) Run Audio Feature Extraction

In [ ]:
subprocess.run([sys.executable, "-m", "src.audio_pipeline"], check=True)\n
audio_features = pd.read_csv(PROCESSED / "audio_features.csv")\n
print("audio_features.csv shape:", audio_features.shape)\n
display(audio_features.head())

## 17) Train All Three Models

In [ ]:
subprocess.run([sys.executable, "-m", "src.train_models"], check=True)\n
print("Training completed.")

## 18) Show Model Evaluation Metrics (Accuracy, F1, Loss)

In [ ]:
metrics_path = REPORTS / "metrics.json"\n
with open(metrics_path, "r", encoding="utf-8") as f:\n
    metrics = json.load(f)\n
\n
print(json.dumps(metrics, indent=2))\n
\n
metrics_rows = []\n
for model_name, vals in metrics.items():\n
    metrics_rows.append({\n
        "model": model_name,\n
        "accuracy": vals.get("accuracy"),\n
        "f1_weighted": vals.get("f1_weighted"),\n
        "loss": vals.get("loss")\n
    })\n
\n
display(pd.DataFrame(metrics_rows))

## 19) Generate Report Visual Assets

In [ ]:
subprocess.run([sys.executable, "-m", "src.visualize_samples"], check=True)\n
print("Saved visuals in reports/.")

## 20) System Demo: Authorized Attempt

In [ ]:
subprocess.run([\n
    sys.executable, "-m", "src.auth_system_cli",\n
    "--face-image", "data/images/member_1/neutral.jpg",\n
    "--voice-audio", "data/audio/member_1/yes_approve.wav"\n
], check=True)

## 21) System Demo: Unauthorized Attempt

In [ ]:
subprocess.run([\n
    sys.executable, "-m", "src.auth_system_cli",\n
    "--face-image", "data/images/unauthorized/unauthorized.jpg",\n
    "--voice-audio", "data/audio/unauthorized/unauthorized voice.wav"\n
], check=True)

## 22) Rubric Coverage Checklist

In [ ]:
checklist = [\n
    ("EDA summary stats + 3 labeled plots", True),\n
    ("Data cleaning + merge validation", True),\n
    ("Image quantity/diversity", True),\n
    ("Image augmentation + feature extraction", True),\n
    ("Audio quality visualization", True),\n
    ("Audio augmentation + feature extraction", True),\n
    ("Three models implemented", True),\n
    ("Accuracy, F1, loss evaluation", True),\n
    ("System simulation authorized + unauthorized", True)\n
]\n
\n
display(pd.DataFrame(checklist, columns=["rubric_item", "covered"]))

## 23) Final Notes\n
\n
- Add short interpretation text after each plot and metric table for maximum rubric score.\n
- Include demo video link, report link, and team contribution summary in final submission.